In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [7]:
print("="*80)
print("IPO EXCESS RETURNS - OUTLIER ANALYSIS")
print("="*80 + "\n")

# Load data
csv_filename = '/Users/danielgarciabruna/Desktop/VScode/Uni/Topics/newcode/ipo_data_with_1year_excess_returns_FINAL.csv'
df = pd.read_csv(csv_filename)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}\n")

IPO EXCESS RETURNS - OUTLIER ANALYSIS

Dataset shape: (881, 37)
Columns: ['Unnamed: 0', 'Issuer.Ticker', 'Issuer.Name', 'Sales...1.Yr.Growth', 'Profit.Margin', 'Return.on.Assets', 'Offer.Size..M.', 'Shares.Outstanding..M.', 'Offer.Price', 'Offer.To.1st.Close', 'Market.Cap.at.Offer..M.', 'Trade.Date..US.', 'cusip', 'Cash.Flow.per.Share', 'Instit.Owner....Shares.Out.', 'Instit.Owner..Shares.Held.', 'Month.Date', 'Month.of.Quarter', 'Real.GDP.Per.Capita', 'OECD.Leading.Indicator', 'Interest.Rate', 'Seasonally.Adjusted.Unemployment.Rate', 'CPI.Growth.Rate', 'Filing.Term.Price.Range', 'Priced.Range', 'Industry.Sector', 'Industry.Group', 'Industry.Subgroup', 'Underpriced', 'price_1yr', 'X1_year_return', 'X1_year_up', 'actual_date_1yr', 'days_from_ipo', 'status', '1_Year_excess_return', '1_day_excess_return']



In [8]:
print("="*80)
print("DESCRIPTIVE STATISTICS")
print("="*80 + "\n")

print("1-DAY EXCESS RETURNS:")
print("-" * 40)
print(df['1_day_excess_return'].describe())
print(f"\nSkewness: {df['1_day_excess_return'].skew():.3f}")
print(f"Kurtosis: {df['1_day_excess_return'].kurtosis():.3f}")

print("\n1-YEAR EXCESS RETURNS:")
print("-" * 40)
print(df['1_Year_excess_return'].describe())
print(f"\nSkewness: {df['1_Year_excess_return'].skew():.3f}")
print(f"Kurtosis: {df['1_Year_excess_return'].kurtosis():.3f}")

DESCRIPTIVE STATISTICS

1-DAY EXCESS RETURNS:
----------------------------------------
count    881.000000
mean      17.450776
std       27.839004
min     -100.797191
25%        0.045817
50%        8.396695
75%       27.016947
max      196.274270
Name: 1_day_excess_return, dtype: float64

Skewness: 1.913
Kurtosis: 5.906

1-YEAR EXCESS RETURNS:
----------------------------------------
count     881.000000
mean       30.816482
std       145.507993
min      -117.453423
25%       -45.865376
50%       -10.510599
75%        52.163389
max      1004.939850
Name: 1_Year_excess_return, dtype: float64

Skewness: 3.394
Kurtosis: 14.959


In [9]:
print("\n" + "="*80)
print("OUTLIER DETECTION")
print("="*80 + "\n")

def detect_outliers(data, column_name):
    """Detect outliers using multiple methods"""
    
    series = data[column_name].dropna()
    
    print(f"\n{'='*60}")
    print(f"{column_name.upper()}")
    print(f"{'='*60}\n")
    
    # Method 1: IQR (Interquartile Range)
    print("METHOD 1: IQR (Interquartile Range)")
    print("-" * 40)
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    iqr_outliers = series[(series < lower_bound) | (series > upper_bound)]
    print(f"Q1 (25th percentile): {Q1:.4f}")
    print(f"Q3 (75th percentile): {Q3:.4f}")
    print(f"IQR: {IQR:.4f}")
    print(f"Lower bound: {lower_bound:.4f}")
    print(f"Upper bound: {upper_bound:.4f}")
    print(f"Number of outliers: {len(iqr_outliers)} ({len(iqr_outliers)/len(series)*100:.1f}%)")
    
    # Method 2: Z-Score
    print("\nMETHOD 2: Z-Score (|z| > 3)")
    print("-" * 40)
    z_scores = np.abs(stats.zscore(series))
    z_outliers = series[z_scores > 3]
    print(f"Mean: {series.mean():.4f}")
    print(f"Std Dev: {series.std():.4f}")
    print(f"Number of outliers: {len(z_outliers)} ({len(z_outliers)/len(series)*100:.1f}%)")
    
    # Method 3: Modified Z-Score (using median)
    print("\nMETHOD 3: Modified Z-Score (|modified_z| > 3.5)")
    print("-" * 40)
    median = series.median()
    mad = np.median(np.abs(series - median))
    modified_z = 0.6745 * (series - median) / mad
    modified_z_outliers = series[np.abs(modified_z) > 3.5]
    print(f"Median: {median:.4f}")
    print(f"MAD (Median Absolute Deviation): {mad:.4f}")
    print(f"Number of outliers: {len(modified_z_outliers)} ({len(modified_z_outliers)/len(series)*100:.1f}%)")
    
    # Method 4: Percentile Method (Beyond 1st/99th percentile)
    print("\nMETHOD 4: Percentile (< 1st or > 99th percentile)")
    print("-" * 40)
    p1 = series.quantile(0.01)
    p99 = series.quantile(0.99)
    percentile_outliers = series[(series < p1) | (series > p99)]
    print(f"1st percentile: {p1:.4f}")
    print(f"99th percentile: {p99:.4f}")
    print(f"Number of outliers: {len(percentile_outliers)} ({len(percentile_outliers)/len(series)*100:.1f}%)")
    
    # Show extreme outliers
    print("\nEXTREME VALUES:")
    print("-" * 40)
    print(f"Top 5 highest values:")
    top_5 = series.nlargest(5)
    for idx, val in top_5.items():
        print(f"  Index {idx}: {val:.4f}")
    
    print(f"\nTop 5 lowest values:")
    bottom_5 = series.nsmallest(5)
    for idx, val in bottom_5.items():
        print(f"  Index {idx}: {val:.4f}")
    
    return {
        'iqr_outliers': iqr_outliers,
        'z_outliers': z_outliers,
        'modified_z_outliers': modified_z_outliers,
        'percentile_outliers': percentile_outliers,
        'bounds': {
            'iqr_lower': lower_bound,
            'iqr_upper': upper_bound,
            'p1': p1,
            'p99': p99
        }
    }

# Detect outliers for both columns
outliers_1day = detect_outliers(df, '1_day_excess_return')
outliers_1year = detect_outliers(df, '1_Year_excess_return')



OUTLIER DETECTION


1_DAY_EXCESS_RETURN

METHOD 1: IQR (Interquartile Range)
----------------------------------------
Q1 (25th percentile): 0.0458
Q3 (75th percentile): 27.0169
IQR: 26.9711
Lower bound: -40.4109
Upper bound: 67.4736
Number of outliers: 50 (5.7%)

METHOD 2: Z-Score (|z| > 3)
----------------------------------------
Mean: 17.4508
Std Dev: 27.8390
Number of outliers: 21 (2.4%)

METHOD 3: Modified Z-Score (|modified_z| > 3.5)
----------------------------------------
Median: 8.3967
MAD (Median Absolute Deviation): 9.7794
Number of outliers: 71 (8.1%)

METHOD 4: Percentile (< 1st or > 99th percentile)
----------------------------------------
1st percentile: -19.4448
99th percentile: 121.1232
Number of outliers: 18 (2.0%)

EXTREME VALUES:
----------------------------------------
Top 5 highest values:
  Index 710: 196.2743
  Index 137: 163.2124
  Index 401: 146.8751
  Index 494: 145.3511
  Index 468: 138.8907

Top 5 lowest values:
  Index 500: -100.7972
  Index 303: -41.1023


In [10]:
print("="*80)
print("DATA CLEANING - CAPPING IMPOSSIBLE VALUES AT -100%")
print("="*80 + "\n")

print("Checking for impossible returns (< -100%)...")

# Check 1-day excess returns (in percentage form)
impossible_1day = df[df['1_day_excess_return'] < -100].copy()
print(f"\n1-Day Excess Returns < -100%: {len(impossible_1day)}")
if len(impossible_1day) > 0:
    print("Impossible values found (will be capped at -100%):")
    print(impossible_1day[['Issuer.Ticker', 'Issuer.Name', '1_day_excess_return']].to_string(index=False))
    # Cap at -100%
    df.loc[df['1_day_excess_return'] < -100, '1_day_excess_return'] = -100
    print(f"✓ Capped {len(impossible_1day)} values to -100%")

# Check 1-year excess returns (in percentage form)
impossible_1year = df[df['1_Year_excess_return'] < -100].copy()
print(f"\n1-Year Excess Returns < -100%: {len(impossible_1year)}")
if len(impossible_1year) > 0:
    print("Impossible values found (will be capped at -100%):")
    print(impossible_1year[['Issuer.Ticker', 'Issuer.Name', '1_Year_excess_return']].to_string(index=False))
    # Cap at -100%
    df.loc[df['1_Year_excess_return'] < -100, '1_Year_excess_return'] = -100
    print(f"✓ Capped {len(impossible_1year)} values to -100%")

if len(impossible_1day) == 0 and len(impossible_1year) == 0:
    print("\n✓ No impossible values found! Data is clean.")
else:
    print(f"\n✓ Data cleaned by capping!")
    print(f"  Total values capped: {len(impossible_1day) + len(impossible_1year)}")
    print(f"  All impossible returns now set to -100%")

DATA CLEANING - CAPPING IMPOSSIBLE VALUES AT -100%

Checking for impossible returns (< -100%)...

1-Day Excess Returns < -100%: 1
Impossible values found (will be capped at -100%):
Issuer.Ticker              Issuer.Name  1_day_excess_return
         MCBS Metrocity Bankshares Inc          -100.797191
✓ Capped 1 values to -100%

1-Year Excess Returns < -100%: 17
Impossible values found (will be capped at -100%):
Issuer.Ticker                    Issuer.Name  1_Year_excess_return
          CMG     Chipotle Mexican Grill Inc           -106.571013
         EARN Ellington Residential Mortgage           -100.368841
           ET             Energy Transfer LP           -103.408287
        GOOGL                   Alphabet Inc           -103.549962
         HAWK Blackhawk Network Holdings Inc           -109.313453
         HOMB         Home BancShares Inc/AR           -102.406043
          LFT       Lument Finance Trust Inc           -105.540350
         LMPX    LMP Automotive Holdings Inc      

In [11]:
print("\n" + "="*80)
print("DESCRIPTIVE STATISTICS (AFTER CLEANING)")
print("="*80 + "\n")

print("1-DAY EXCESS RETURNS:")
print("-" * 40)
print(df['1_day_excess_return'].describe())
print(f"\nSkewness: {df['1_day_excess_return'].skew():.3f}")
print(f"Kurtosis: {df['1_day_excess_return'].kurtosis():.3f}")

print("\n1-YEAR EXCESS RETURNS:")
print("-" * 40)
print(df['1_Year_excess_return'].describe())
print(f"\nSkewness: {df['1_Year_excess_return'].skew():.3f}")
print(f"Kurtosis: {df['1_Year_excess_return'].kurtosis():.3f}")


DESCRIPTIVE STATISTICS (AFTER CLEANING)

1-DAY EXCESS RETURNS:
----------------------------------------
count    881.000000
mean      17.451681
std       27.835169
min     -100.000000
25%        0.045817
50%        8.396695
75%       27.016947
max      196.274270
Name: 1_day_excess_return, dtype: float64

Skewness: 1.915
Kurtosis: 5.901

1-YEAR EXCESS RETURNS:
----------------------------------------
count     881.000000
mean       30.936047
std       145.396415
min      -100.000000
25%       -45.865376
50%       -10.510599
75%        52.163389
max      1004.939850
Name: 1_Year_excess_return, dtype: float64

Skewness: 3.402
Kurtosis: 15.000


In [12]:

# Save to new CSV file
output_filename = '/Users/danielgarciabruna/Desktop/VScode/Uni/Topics/newcode/ipo_data_with_1year_excess_returns_CLEANED.csv'
df.to_csv(output_filename, index=False)
print(f"✓ Cleaned data saved to:")
print(f"  {output_filename}")
print(f"\n  Total rows: {len(df)}")
print(f"  Total columns: {len(df.columns)}")

✓ Cleaned data saved to:
  /Users/danielgarciabruna/Desktop/VScode/Uni/Topics/newcode/ipo_data_with_1year_excess_returns_CLEANED.csv

  Total rows: 881
  Total columns: 37
